In [1]:
import numpy as np 
import pandas as pd 
#import os
#for dirname, _, filenames in os.walk('/kaggle/input'):
    #for filename in filenames:
        #print(os.path.join(dirname, filen|ame))


In [2]:
train = pd.read_csv('../input/osic-pulmonary-fibrosis-progression/train.csv')
test = pd.read_csv('../input/osic-pulmonary-fibrosis-progression/test.csv')


In [3]:
from tqdm import tqdm
#  also cnsider back time
train_exp = pd.DataFrame()

for patient in tqdm(train.Patient.unique()):
    df = train.loc[train.Patient == patient,:]

    for idx,week,percent in zip(df.index,df.Weeks,df.Percent):
        
        temp_df_pos            = df.loc[idx:,:'SmokingStatus']
        temp_df_pos['Percent'] = percent
        temp_df_pos['Weeks']   = week
        temp_df_pos['target']  = temp_df_pos['FVC']
        temp_df_pos['delta']   = df.loc[idx:,'Weeks'] - df.loc[idx,'Weeks']
        temp_df_pos['FVC']     = temp_df_pos.loc[idx,'FVC']
        
        
        temp_df_neg            = df.loc[:idx,:'SmokingStatus']
        temp_df_neg['Weeks']   = week
        temp_df_neg['Percent'] = percent
        temp_df_neg['target']  = temp_df_neg['FVC']
        temp_df_neg['delta']   = df.loc[:idx,'Weeks'] - df.loc[idx,'Weeks']
        temp_df_neg['FVC']     = temp_df_neg.loc[idx,'FVC']
        
        train_exp = pd.concat([train_exp,temp_df_pos,temp_df_neg],axis = 0)
        train_exp = train_exp[train_exp.delta!=0].drop_duplicates().dropna(axis = 0).reset_index(drop =True)        


100%|██████████| 158/158 [00:06<00:00, 24.72it/s]


In [4]:
from sklearn.model_selection import cross_val_score, train_test_split, GridSearchCV,GroupKFold
from sklearn.pipeline        import Pipeline, make_pipeline
from sklearn.compose         import ColumnTransformer, make_column_transformer
from sklearn.metrics         import mean_squared_error,mean_absolute_error

from sklearn.preprocessing   import OneHotEncoder,OrdinalEncoder
from sklearn.preprocessing   import MinMaxScaler,StandardScaler,RobustScaler
from sklearn.preprocessing   import FunctionTransformer
from sklearn.ensemble        import RandomForestRegressor,ExtraTreesRegressor,GradientBoostingRegressor
from sklearn.ensemble        import StackingRegressor
from sklearn.linear_model    import LinearRegression
from sklearn.naive_bayes     import MultinomialNB

from sklearn.svm             import SVR

X = train_exp.drop(['Patient','target'],axis = 1)
y = train_exp['target']


X_train,X_val,y_train,y_val = train_test_split(X,y,
                                               test_size     = 0.2,
                                               random_state  = 42,
                                               shuffle       = True)



transformer = make_column_transformer(
    (MinMaxScaler() , ['FVC','Percent','Age','Weeks','delta']),
    (OrdinalEncoder(),['Sex','SmokingStatus']),
    remainder = 'passthrough'
    
)



pipeline = make_pipeline(transformer,
                         RandomForestRegressor(n_estimators  = 300,
                                               max_depth     = 5))





score = np.sqrt(-cross_val_score(pipeline,X_train,y_train,cv=5, scoring = 'neg_mean_squared_error'))

#print('Score: ',score.mean())
#print('STD  : ', score.std())

pipeline.fit(X_train,y_train)
#param_grid = {'randomforestregressor__n_estimators':[50,100,500],
              #'randomforestregressor__max_depth':[5,10,20],
              #'randomforestregressor__max_features':[2,5,7],
              #'randomforestregressor__min_samples_leaf' : [3,6,10]}

#search = GridSearchCV(pipeline,param_grid, cv=5,scoring = 'neg_mean_squared_error')
#search.fit(X_train,y_train)

#print('Best score:', np.sqrt(-search.best_score_))
#print('Best param:', search.best_params_)
#pipeline = search.best_estimator_


Pipeline(steps=[('columntransformer',
                 ColumnTransformer(remainder='passthrough',
                                   transformers=[('minmaxscaler',
                                                  MinMaxScaler(),
                                                  ['FVC', 'Percent', 'Age',
                                                   'Weeks', 'delta']),
                                                 ('ordinalencoder',
                                                  OrdinalEncoder(),
                                                  ['Sex', 'SmokingStatus'])])),
                ('randomforestregressor',
                 RandomForestRegressor(max_depth=5, n_estimators=300))])

In [5]:
def confidence(pipe,regressor,X_val,transformer):
    
    val = transformer.transform(X_val)
    predictions = []
    for tree in pipe[regressor]:
        predictions.append(tree.predict(val))

    confidence = np.std(predictions,axis=0)
    return confidence


def laplace_log_likelihood(actual_fvc, predicted_fvc, confidence, return_values = False):
    """
    Calculates the modified Laplace Log Likelihood score for this competition.
    """
    sd_clipped = np.maximum(confidence, 70)
    delta = np.minimum(np.abs(actual_fvc - predicted_fvc), 1000)
    metric = - np.sqrt(2) * delta / sd_clipped - np.log(np.sqrt(2) * sd_clipped)

    if return_values:
        return metric
    else:
        return np.mean(metric)

def pred_ints(model, X, percentile=.95):
    
    err_down = []
    err_up = []
    for x in range(len(X)):
        preds = []
        for pred in model['randomforestregressor'].estimators_:
            preds.append(pred.predict(X[x].reshape(1,-1))[0])
        err_down.append(np.percentile(preds, (100 - percentile) / 2. ))
        err_up.append(np.percentile(preds, 100 - (100 - percentile) / 2.))
        
    return err_down, err_up

print('Train OSCI score: ',laplace_log_likelihood(y_train, pipeline.predict(X_train), confidence(pipeline,'randomforestregressor',X_train,transformer), return_values = False))
print('Val   OSCI score: ',laplace_log_likelihood(y_val, pipeline.predict(X_val), confidence(pipeline,'randomforestregressor',X_val,transformer), return_values = False))

print('Train RMSE score: ',np.sqrt(mean_squared_error(y_train, pipeline.predict(X_train))))
print('Val   RMSE score: ',np.sqrt(mean_squared_error(y_val, pipeline.predict(X_val))))


Train OSCI score:  -7.326884857806743
Val   OSCI score:  -7.319791084034745
Train RMSE score:  215.8195421707593
Val   RMSE score:  213.59117573184514


In [6]:
new_test = pd.DataFrame()
for i in np.arange(-12,134,1):
    temp_df = test.copy()
    temp_df['stamps'] = i
    temp_df['delta'] = temp_df['Weeks'] +  temp_df['stamps']
    new_test = pd.concat([new_test,temp_df])

new_test.reset_index(drop=True,inplace = True)
new_test['Patient_Week'] = new_test['Patient'] + '_' + new_test['stamps'].astype(str)


X_test = new_test.drop(['Patient','stamps','Patient_Week'],axis = 1)


In [7]:
#confidence        = confidence(pipeline,'randomforestregressor',X_test,transformer)
#predictions       = pipeline.predict(X_test)
#submission        = pd.DataFrame({'Patient_Week':new_test.Patient_Week,'FVC':predictions,'Confidence':confidence})

#submission.to_csv('submission.csv',
                  #index = False)


In [8]:
#submission


In [9]:
X = train_exp.drop(['Patient','target'],axis = 1)
y = train_exp['target']


X_train,X_val,y_train,y_val = train_test_split(X,y,
                                               test_size     = 0.2,
                                               random_state  = 42,
                                               shuffle       = True)



transformer = make_column_transformer(
    (MinMaxScaler() , ['FVC','Percent','Age','Weeks','delta']),
    (OrdinalEncoder(),['Sex','SmokingStatus']),
    remainder = 'passthrough'
    
)


X_train = transformer.fit_transform(X_train)
X_val   = transformer.transform(X_val)

alpha = 0.95

clf = GradientBoostingRegressor(loss='quantile', alpha=alpha,
                                n_estimators=250, max_depth=3,
                                learning_rate=.1, min_samples_leaf=9,
                                min_samples_split=9)


clf.fit(X_train, y_train)
y_upper_train = clf.predict(X_train)
y_upper_val = clf.predict(X_val)

clf.set_params(alpha=1.0 - alpha)
clf.fit(X_train, y_train)
y_lower_train = clf.predict(X_train)
y_lower_val = clf.predict(X_val)


clf.set_params(loss='ls')
clf.fit(X_train, y_train)
y_pred_train = clf.predict(X_train)
y_pred_val = clf.predict(X_val)



print('Train RMSE score: ',np.sqrt(mean_squared_error(y_train, y_pred_train)))
print('Val   RMSE score: ',np.sqrt(mean_squared_error(y_val, y_pred_val)))


confidence_train = (y_upper_train - y_lower_train)/2
confidence_val = (y_upper_val - y_lower_val)/2


print('Train OSCI score: ',laplace_log_likelihood(y_train, y_pred_train, confidence_train, return_values = False))
print('Val   OSCI score: ',laplace_log_likelihood(y_val, y_pred_val, confidence_val, return_values = False))


InvalidParameterError: The 'loss' parameter of GradientBoostingRegressor must be a str among {'squared_error', 'huber', 'quantile', 'absolute_error'}. Got 'ls' instead.

In [10]:
new_test = pd.DataFrame()
for i in np.arange(-12,134,1):
    temp_df = test.copy()
    temp_df['stamps'] = i
    temp_df['delta'] = temp_df['Weeks'] +  temp_df['stamps']
    new_test = pd.concat([new_test,temp_df])

new_test.reset_index(drop=True,inplace = True)
new_test['Patient_Week'] = new_test['Patient'] + '_' + new_test['stamps'].astype(str)


X_test = new_test.drop(['Patient','stamps','Patient_Week'],axis = 1)


X_test = transformer.transform(X_test)

alpha = 0.95

clf = GradientBoostingRegressor(loss='quantile', alpha=alpha,
                                n_estimators=250, max_depth=3,
                                learning_rate=.1, min_samples_leaf=9,
                                min_samples_split=9)


clf.fit(X_train, y_train)
y_upper_train = clf.predict(X_train)
y_upper_test = clf.predict(X_test)

clf.set_params(alpha=1.0 - alpha)

clf.fit(X_train, y_train)
y_lower_train = clf.predict(X_train)
y_lower_test = clf.predict(X_test)


clf.set_params(loss='ls')
clf.fit(X_train, y_train)
y_pred_train = clf.predict(X_train)
y_pred_test = clf.predict(X_test)\


confidence_train = (y_upper_train - y_lower_train)/2
confidence_test = (y_upper_test - y_lower_test)/2

confidence        = confidence_test
predictions       = y_pred_test = clf.predict(X_test)

submission        = pd.DataFrame({'Patient_Week':new_test.Patient_Week,'FVC':predictions,'Confidence':confidence})

submission.to_csv('submission.csv',
                  index = False)


InvalidParameterError: The 'loss' parameter of GradientBoostingRegressor must be a str among {'squared_error', 'huber', 'quantile', 'absolute_error'}. Got 'ls' instead.

In [11]:
submission


NameError: name 'submission' is not defined